In [24]:
import requests as rq
from lxml import etree
import pandas as pd

In [25]:
url = 'https://opendata.fmi.fi/wfs'
params = {
    'service':'WFS',
    'version':'2.0.0',
    'request':'getFeature',
    'storedquery_id':'fmi::forecast::harmonie::surface::point::multipointcoverage',
    'place':'helsinki',
    # 'starttime': '',
    # 'endtime':'',
}

In [26]:
resp = rq.get(url=url, params=params)
resp.raise_for_status

<bound method Response.raise_for_status of <Response [200]>>

In [27]:
root = etree.fromstring(resp.text.encode())
ns = root.nsmap
rows = []
columns1 = []
time_stamps = []


In [ ]:
# columns
alist = [x.get('name').lower() for x in root.xpath('//swe:field', namespaces=ns)]
columns1 = alist


for num, tvp in enumerate(root.xpath('//gml:doubleOrNilReasonTupleList', namespaces=ns)):
    alist = tvp.text.split('\n')
    arow = [x.strip().replace(' ', ',') for x in alist if x != '']
    rows = arow
    # print(rows[49])
rows.pop(50)

# timestamps
for num, tvp in enumerate(root.xpath('//gmlcov:positions', namespaces=ns)):
    alist = tvp.text.split('\n')
    time_stamps = [x.strip().replace(' ', ',').replace(',,', ',').replace('60.16952,24.93545,', '') for x in alist if x != '']
    # print(time_stamps)
time_stamps.pop(50)

''

In [29]:
ready_rows = []
for x in rows:
    alist = [y for y in x.split(',')]
    ready_rows.append(alist)

In [30]:

# print(new_rows)
df1 = pd.DataFrame(ready_rows, columns=columns1, dtype=float)


In [31]:
df1['timestamps'] = time_stamps

In [32]:
df1['timestamps'] = pd.to_datetime(df1['timestamps'].astype(int), unit='s')

# df1.head()

In [33]:
df2 = df1.drop(columns=['radiationnetsurfacelwaccumulation'])
# df2.head()

In [34]:
df3 = df2.drop_duplicates()

In [35]:
df3.columns


Index(['pressure', 'geopheight', 'temperature', 'dewpoint', 'humidity',
       'winddirection', 'windspeedms', 'windums', 'windvms',
       'precipitationamount', 'totalcloudcover', 'lowcloudcover',
       'mediumcloudcover', 'highcloudcover', 'radiationglobal',
       'radiationglobalaccumulation', 'radiationnetsurfaceswaccumulation',
       'radiationswaccumulation', 'visibility', 'windgust', 'timestamps'],
      dtype='str')